# 04 · Credibility checks for Q2

Each check changes one setting of `03_q2`, reruns the same steps (shared functions in `q2_tiers.py`) and
compares the result with the main analysis. The question each time: **do the focus tiers keep the same
sellers?**

| # | Check | What changes |
|---|---|---|
| 1 | Minimum orders | 10 and 30 instead of 20 |
| 2 | Lateness definition | by timestamp instead of by calendar date |
| 3 | Orders handed over before payment approval | the 1,156 group B orders added back |
| 4 | Number of tiers | k = 3 and k = 5 instead of 4 |
| 5 | Earlier features | log(order volume) instead of log(1 + late orders) |
| 6 | 2017 → 2018 holdout | tiers built on 2017; are 2017 focus sellers still worse in 2018, and do sellers stay in the same tier? |
| 7 | Which review | first instead of last review |
| 8 | Product category | on-time ratings compared within the same category |
| 9 | Chance | are the top 20 sellers worse than chance, allowing for months that bunch together? |
| 10 | Size of the effect | what fixing the top sellers could mean for late deliveries and low ratings |

**Summary of results**

- **Threshold, lateness definition, group B orders (checks 1–3):** at least 99% of the main focus sellers stay in focus, and the top 20 sellers stay in focus in every one of these checks.
- **Number of tiers (check 4):** k = 5 gives almost the same focus sellers (97%). With k = 3 the moderate tier merges into a lower tier, so only the 58 most severe sellers are in focus (36% of the main focus sellers, but 90% of the top 20). With three tiers, 18 of the original top 20 sellers stay in focus. This check measures focus membership rather than rank.
- **Earlier features (check 5):** with order volume the seller with the most excess late handovers is not in focus, and only 47% of the original focus sellers stay in focus, including 14 of the original top 20. This supports using the late handover count.
- **Holdout (check 6):** sellers in focus on 2017 data are still worse in 2018 (late rate 7.3% vs 2.6%; +1.7 vs −2.9 percentage points against the 2018 same-month benchmark; 264 sellers active in both years; Spearman 0.57). Individual sellers move, though: with tiers built separately for each year, 51% of sellers stay in the same tier and 62% of 2017 focus sellers are still in focus in 2018. The 2017 Tier 4 sellers fall from 33% to 9.5% late, likely partly because a small seller's bad year is not repeated. The tiers are a snapshot of one period, so sellers should be re-scored each period with fixed boundaries.
- **First vs last review (check 7):** no difference.
- **Product category (check 8):** on orders that were on time at both steps, only Tier 4 is rated below its category average (−0.34 stars); Tiers 1–3 are within ±0.07.
- **Chance (check 9):** late handovers bunch in months (dispersion ratio about 2). Allowing for that, all top 20 sellers, and 74 sellers in total, are worse than the platform; all 74 are in the focus tiers.
- **Size of the effect (check 10):** only about 20% of late deliveries followed a late handover. Fixing the top 20 sellers' excess late handovers might avoid about 3% of late deliveries and about 1% of low ratings (association-based estimate).

## 0. Settings and the main result

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

from q2_tiers import add_excess, assign_tiers, focus_tiers, profile_tiers, seller_metrics_from_orders

MIN_ORDERS = 20
CHOSEN_K = 4
RATING_COL = "review_score_last"
LOW_RATING = 2
ALPHA = 0.05

sellers_main = pd.read_csv("../Data/Clean/seller_metrics.csv")
orders_main = pd.read_csv(
    "../Data/Clean/orders_main.csv",
    parse_dates=["order_delivered_carrier_date", "shipping_limit_date",
                 "order_delivered_customer_date", "order_estimated_delivery_date"],
)
orders_group_b = pd.read_csv("../Data/Clean/orders_group_b.csv")


def run_tiers(seller_table, k=CHOSEN_K, min_orders=MIN_ORDERS, scale="late_orders"):
    """Run the 03_q2 steps and return the tiered sellers (with a focus flag) and the profile."""
    tiers = add_excess(assign_tiers(seller_table, k, min_orders, scale))
    profile = profile_tiers(tiers)
    tiers["focus_tier"] = tiers["tier"].isin(focus_tiers(profile))
    return tiers, profile


main_tiers, main_profile = run_tiers(sellers_main)
main_focus = set(main_tiers.loc[main_tiers["focus_tier"], "seller_id"])
top_seller = main_tiers.loc[main_tiers["excess_late"].idxmax(), "seller_id"]
main_top20 = set(main_tiers[main_tiers["focus_tier"]].nlargest(20, "excess_late")["seller_id"])
print(f"Main result: {len(main_tiers)} sellers, {len(main_focus)} in focus tiers")

Main result: 778 sellers, 159 in focus tiers


## 1–5. Do the focus tiers keep the same sellers?

For each variant: how many sellers are clustered, how many fall in the focus tiers, what share of late
handovers those tiers hold, what share of the main focus sellers stay in focus (counting only sellers that
are clustered in both), what share of the main top 20 sellers (by excess late handovers) stay in focus, and
whether the seller with the most excess late handovers is still in focus.

In [2]:
def compare_with_main(name, tiers):
    """One summary row comparing a variant's focus sellers with the main result."""
    focus = set(tiers.loc[tiers["focus_tier"], "seller_id"])
    shared_main_focus = main_focus & set(tiers["seller_id"])
    return {
        "check": name,
        "sellers_clustered": len(tiers),
        "focus_sellers": len(focus),
        "late_share_in_focus": tiers.loc[tiers["focus_tier"], "late_orders"].sum() / tiers["late_orders"].sum(),
        "main_focus_kept": len(shared_main_focus & focus) / len(shared_main_focus),
        "new_in_focus": len(focus - main_focus),
        "main_top20_kept": len(main_top20 & focus) / len(main_top20 & set(tiers["seller_id"])),
        "top_seller_in_focus": top_seller in focus,
    }


# Lateness by timestamp instead of by calendar date
orders_ts = orders_main.assign(
    late_ts=orders_main["order_delivered_carrier_date"] > orders_main["shipping_limit_date"]
)
sellers_timestamp = seller_metrics_from_orders(orders_ts, late_col="late_ts")

# Main sample plus the orders handed over before payment approval
columns = ["seller_id", "order_purchase_month", "late_handover"]
sellers_with_b = seller_metrics_from_orders(pd.concat([orders_main[columns], orders_group_b[columns]]))

variants = {
    "Main (20 orders, by date, k = 4)": run_tiers(sellers_main)[0],
    "1a Minimum 10 orders": run_tiers(sellers_main, min_orders=10)[0],
    "1b Minimum 30 orders": run_tiers(sellers_main, min_orders=30)[0],
    "2 Late by timestamp": run_tiers(sellers_timestamp)[0],
    "3 Group B orders added": run_tiers(sellers_with_b)[0],
    "4a k = 3": run_tiers(sellers_main, k=3)[0],
    "4b k = 5": run_tiers(sellers_main, k=5)[0],
    "5 Earlier features (order volume)": run_tiers(sellers_main, scale="order_volume")[0],
}
check_table = pd.DataFrame([compare_with_main(name, t) for name, t in variants.items()]).set_index("check")
check_table.style.format({"late_share_in_focus": "{:.0%}", "main_focus_kept": "{:.0%}", "main_top20_kept": "{:.0%}"})

,sellers_clustered,focus_sellers,late_share_in_focus,main_focus_kept,new_in_focus,main_top20_kept,top_seller_in_focus
check,,,,,,,
"Main (20 orders, by date, k = 4)",778,159,77%,100%,0,100%,True
1a Minimum 10 orders,1199,242,81%,99%,84,100%,True
1b Minimum 30 orders,604,147,79%,100%,4,100%,True
2 Late by timestamp,778,224,83%,99%,66,100%,True
3 Group B orders added,784,163,78%,100%,4,100%,True
4a k = 3,778,58,37%,36%,0,90%,True
4b k = 5,778,166,79%,97%,11,100%,True
5 Earlier features (order volume),778,75,32%,47%,0,70%,False


## 6. Holdout: build tiers on 2017, check the same sellers in 2018

Tiers are built from 2017 orders only (same steps, benchmark from 2017 months). For sellers with at least
20 orders in both years, the 2018 late rate and the gap to the 2018 same-month benchmark are compared
between 2017 focus sellers and the rest. If 2017 focus sellers are still worse in 2018, the tiers capture
lasting behaviour rather than one bad period. This shows persistence, not that acting on the list would
help.

In [3]:
months = orders_main["order_purchase_month"].astype(str)
sellers_2017 = seller_metrics_from_orders(orders_main[months.str.startswith("2017")])
sellers_2018 = seller_metrics_from_orders(orders_main[months.str.startswith("2018")])

tiers_2017, profile_2017 = run_tiers(sellers_2017)
both_years = tiers_2017.merge(
    sellers_2018[sellers_2018["order_volume"] >= MIN_ORDERS], on="seller_id", suffixes=("_2017", "_2018")
)
print(f"Sellers clustered on 2017: {len(tiers_2017)}; with at least {MIN_ORDERS} orders in both years: {len(both_years)}")


def year_2018_summary(df):
    """2018 late rate and gap to the 2018 same-month benchmark, weighted by orders."""
    orders = df["order_volume_2018"].sum()
    late = df["late_orders_2018"].sum()
    expected = (df["benchmark_late_rate_2018"] * df["order_volume_2018"]).sum()
    return pd.Series({"sellers": len(df), "late_rate_2018": late / orders,
                      "gap_vs_same_month_pp_2018": (late - expected) / orders * 100})


holdout = both_years.groupby("focus_tier").apply(year_2018_summary, include_groups=False)
holdout.index = ["Other 2017 sellers", "2017 focus sellers"]
rho = stats.spearmanr(both_years["late_rate_2017"], both_years["late_rate_2018"]).correlation
print(f"Spearman correlation of seller late rates, 2017 vs 2018: {rho:.2f}")
holdout.style.format({"sellers": "{:.0f}", "late_rate_2018": "{:.1%}", "gap_vs_same_month_pp_2018": "{:+.1f}"})

Sellers clustered on 2017: 420; with at least 20 orders in both years: 264
Spearman correlation of seller late rates, 2017 vs 2018: 0.57


,sellers,late_rate_2018,gap_vs_same_month_pp_2018
Other 2017 sellers,177,2.6%,-2.9
2017 focus sellers,87,7.3%,+1.7


### 6b. Do individual sellers stay in the same tier?

The table above compares groups. Here tiers are also built on 2018 orders alone (same steps), and each
seller clustered in both years is followed from its 2017 tier (rows) to its 2018 tier (columns). The
second table shows each 2017 tier's late rate in 2017 and in 2018.

In [4]:
tiers_2018, profile_2018 = run_tiers(sellers_2018)
moves = tiers_2017[["seller_id", "tier", "focus_tier"]].merge(
    tiers_2018[["seller_id", "tier", "focus_tier"]], on="seller_id", suffixes=("_2017", "_2018")
)
focus_2017 = moves[moves["focus_tier_2017"]]
print(f"Sellers clustered in both years: {len(moves)}")
print(f"Same tier in both years: {(moves['tier_2017'] == moves['tier_2018']).mean():.0%}")
print(f"2017 focus sellers still in focus in 2018: {focus_2017['focus_tier_2018'].sum()} of {len(focus_2017)} "
      f"({focus_2017['focus_tier_2018'].mean():.0%})")
display(pd.crosstab(moves["tier_2017"], moves["tier_2018"], margins=True, margins_name="Total"))

# Late rate of each 2017 tier in both years (same sellers, weighted by orders)
totals = both_years.groupby("tier")[["late_orders_2017", "order_volume_2017", "late_orders_2018", "order_volume_2018"]].sum()
pd.DataFrame({
    "sellers": both_years.groupby("tier").size(),
    "late_rate_2017": totals["late_orders_2017"] / totals["order_volume_2017"],
    "late_rate_2018": totals["late_orders_2018"] / totals["order_volume_2018"],
}).style.format({"late_rate_2017": "{:.1%}", "late_rate_2018": "{:.1%}"})

Sellers clustered in both years: 264
Same tier in both years: 51%
2017 focus sellers still in focus in 2018: 54 of 87 (62%)


tier_2018,1,2,3,4,Total
tier_2017,,,,,
1,45,44,6,0,95
2,18,46,15,3,82
3,4,22,40,7,73
4,2,5,4,3,14
Total,69,117,65,13,264


,sellers,late_rate_2017,late_rate_2018
tier,,,
1,95,0.0%,1.9%
2,82,2.0%,3.2%
3,73,7.0%,7.0%
4,14,33.2%,9.5%


Groups persist, but individual sellers move between tiers. Part of the movement is noise: a seller with a
few dozen orders can have one bad year that is not repeated (regression to the mean), which is why the most
severe 2017 tier improves the most. The tiers are therefore a snapshot of one period. In practice sellers
would be re-scored each period on a window of the same length (for example the last 6 months), using the
boundaries fitted here rather than refitting K-means, and stronger action would wait for a seller to stay in
a high tier for two periods in a row.

## 7. First versus last review

In [5]:
rated = orders_main.merge(main_tiers[["seller_id", "tier"]], on="seller_id")
review_table = pd.DataFrame({
    f"mean_{col}": rated.groupby("tier")[col].mean() for col in ["review_score_last", "review_score_first"]
})
for col in ["review_score_last", "review_score_first"]:
    review_table[f"low_share_{col}"] = rated.groupby("tier")[col].apply(lambda s: (s.dropna() <= LOW_RATING).mean())
differing = (rated["review_score_first"] != rated["review_score_last"]) & rated["review_score_first"].notna()
print(f"Orders of clustered sellers whose first and last review differ: {differing.sum()}")
review_table.round(3)

Orders of clustered sellers whose first and last review differ: 166


,mean_review_score_last,mean_review_score_first,low_share_review_score_last,low_share_review_score_first
tier,,,,
1,4.299,4.300,0.096,0.096
2,4.186,4.187,0.118,0.118
3,4.088,4.089,0.141,0.141
4,3.628,3.630,0.260,0.260


## 8. On-time ratings within the same product category

Do higher tiers still get lower ratings on orders that were on time at both steps once product category is
held fixed? Each on-time order's rating is compared with the average on-time rating of its category. The
category comes from the raw product table (the order's first item); this is a merge for the check, not a
cleaning step.

In [6]:
items = pd.read_csv("../Data/Raw/olist_order_items_dataset.csv", usecols=["order_id", "product_id"])
products = pd.read_csv("../Data/Raw/olist_products_dataset.csv", usecols=["product_id", "product_category_name"])
order_category = items.merge(products, on="product_id").groupby("order_id")["product_category_name"].first()

rated["category"] = rated["order_id"].map(order_category).fillna("unknown")
rated["late_delivery"] = (
    rated["order_delivered_customer_date"].dt.normalize() > rated["order_estimated_delivery_date"].dt.normalize()
)
on_time = rated[~rated["late_handover"] & ~rated["late_delivery"]].dropna(subset=[RATING_COL]).copy()
on_time["vs_category"] = on_time[RATING_COL] - on_time.groupby("category")[RATING_COL].transform("mean")
on_time.groupby("tier")["vs_category"].agg(["size", "mean"]).round(2)

,size,mean
tier,,
1,19462,0.07
2,25875,-0.01
3,25817,-0.02
4,1377,-0.34


## 9. Are the top 20 sellers worse than chance?

A binomial test assumes each order is late or not independently. In practice a seller's late handovers
bunch in some months, so the spread is wider than the binomial allows. The **dispersion ratio** measures by
how much (1 = binomial). The test is widened by that ratio, and p-values are adjusted for testing all
clustered sellers at once (Benjamini–Hochberg, 5% false discovery rate).

In [7]:
monthly = orders_main.groupby(["seller_id", "order_purchase_month"]).agg(
    n=("late_handover", "size"), late=("late_handover", "sum")
).reset_index().merge(main_tiers[["seller_id", "late_rate"]], on="seller_id")
monthly = monthly[(monthly["late_rate"] > 0) & (monthly["late_rate"] < 1)]
pearson_chi2 = ((monthly["late"] - monthly["n"] * monthly["late_rate"]) ** 2
                / (monthly["n"] * monthly["late_rate"] * (1 - monthly["late_rate"]))).sum()
dispersion = pearson_chi2 / (len(monthly) - monthly["seller_id"].nunique())
print(f"Dispersion ratio: {dispersion:.2f}")


def benjamini_hochberg(p_values):
    """Return Benjamini-Hochberg adjusted p-values in the original order."""
    p = np.asarray(p_values)
    order = np.argsort(p)
    ranked = p[order] * len(p) / np.arange(1, len(p) + 1)
    q = np.empty_like(ranked)
    q[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1)
    return q


t = main_tiers
z = t["excess_late"] / np.sqrt(dispersion * t["expected_late"] * (1 - t["benchmark_late_rate"]))
t = t.assign(q_value=benjamini_hochberg(stats.norm.sf(z)))
t["worse_than_chance"] = (t["q_value"] < ALPHA) & (t["excess_late"] > 0)
top20 = t[t["focus_tier"]].sort_values("excess_late", ascending=False).head(20)
print(f"Top 20 focus sellers worse than chance: {top20['worse_than_chance'].sum()} of 20")
print(f"All clustered sellers worse than chance: {t['worse_than_chance'].sum()}, "
      f"all in focus tiers: {t.loc[t['worse_than_chance'], 'focus_tier'].all()}")

Dispersion ratio: 1.97
Top 20 focus sellers worse than chance: 20 of 20
All clustered sellers worse than chance: 74, all in focus tiers: True


## 10. Size of the effect on customers

A rough, association-based estimate: if the top 20 focus sellers' excess late handovers were handed over on
time, how many late deliveries and low ratings might be avoided? It uses the difference in late-delivery
rates between late and on-time handovers, and the difference in low ratings between late and on-time
deliveries. It is not a causal estimate.

In [8]:
orders_tiered = orders_main.merge(main_tiers[["seller_id"]], on="seller_id")
orders_tiered["late_delivery"] = (
    orders_tiered["order_delivered_customer_date"].dt.normalize()
    > orders_tiered["order_estimated_delivery_date"].dt.normalize()
)
p_late_delivery = orders_tiered.groupby("late_handover")["late_delivery"].mean()
reviewed = orders_tiered.dropna(subset=[RATING_COL])
p_low = reviewed.groupby("late_delivery")[RATING_COL].apply(lambda s: (s <= LOW_RATING).mean())

excess_top20 = top20["excess_late"].sum()
avoided_late_deliveries = excess_top20 * (p_late_delivery[True] - p_late_delivery[False])
avoided_low_ratings = avoided_late_deliveries * (p_low[True] - p_low[False])
print(f"Late deliveries that followed a late handover: "
      f"{orders_tiered.loc[orders_tiered['late_delivery'], 'late_handover'].mean():.0%}")
print(f"Top 20 excess late handovers: {excess_top20:.0f}")
print(f"Possible avoided late deliveries: about {avoided_late_deliveries:.0f} "
      f"of {orders_tiered['late_delivery'].sum():,} ({avoided_late_deliveries / orders_tiered['late_delivery'].sum():.1%})")
print(f"Possible avoided low ratings: about {avoided_low_ratings:.0f} "
      f"of {(reviewed[RATING_COL] <= LOW_RATING).sum():,}")

Late deliveries that followed a late handover: 20%
Top 20 excess late handovers: 898
Possible avoided late deliveries: about 179 of 5,663 (3.2%)
Possible avoided low ratings: about 97 of 10,167
